# Chapter 15 — Supplement 3: `search_policy`

*Retrieve on the graph, generate on the model.*

Companion to `15_capstone.ipynb`. `search_policy` answers a policy question by **retrieving** the governing policy from a trained kg-memory (GMS) store and then having Qwen **synthesize** an answer constrained to the retrieved text. This notebook calls the shipped retriever and shows both halves.

## Where it fits

Step 3 of the workflow. Given a query (often built from the extracted issue), it returns the governing policy id and a one- or two-sentence grounded answer.

| | |
| --- | --- |
| **Backed by** | Graph RAG over a kg-memory policy store + Qwen grounded synthesis |
| **Artifact** | `data/gms_policy_store/` (trained store) + `data/policies/*.txt` |
| **Build script** | `scripts/build_policy_rag_store.py` (source: `data/bank_policies.md`) |
| **Module** | `glassloop/capstone/policy_rag.py` |
| **Offline toggle** | `AGENTLAB_USE_LLM_RAG=0` -> retrieval only, no synthesis |
| **Risk level** | `LOW` |

The division of labour (chapter §"The five tools"): *retrieval is the graph's job; generation is the model's; grounding is enforced by feeding the model only what the graph returned.*

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. Load the retriever

`get_default_retriever()` loads the trained GMS policy store from `data/gms_policy_store/` and (by default) a Qwen synthesis backend. The store is the geometric kg-memory of Chapter 9 — graph-plus-vector retrieval, not a flat embedding index.

In [ ]:
from glassloop.capstone.policy_rag import get_default_retriever

retriever = get_default_retriever()
print('store loaded :', retriever.store is not None)
print('synthesis on :', retriever.synthesize)
print('policies dir :', retriever.policies_dir)

## 2. The alias lexicon — routing customer vocabulary

Customers do not speak in policy ids. The store learned a `has_alias` edge for every customer phrase, and the retriever turns those into an alias lexicon: `chargeback -> disputes`, `nsf -> overdraft`, and so on. A query is alias-expanded before retrieval so the graph can route vocabulary it would otherwise miss.

In [ ]:
for alias, policy in sorted(retriever.alias_to_policy.items()):
    print(f'  {alias:28s} -> {policy}')
print('\nexpansion example:')
print('  ', retriever._expand_query('I want to file a chargeback'))

The query `"I want to file a chargeback"` is expanded with the routed policy id `disputes`, so retrieval lands on the right policy even though the word *disputes* never appeared in the customer's text. This is the alias-expansion step the chapter highlights.

## 3. Retrieve + synthesize

`retriever.search(query, k=3)` returns the routed policy id, the retrieved policy text, and a Qwen answer **grounded** in that text — the system prompt forbids inventing any fee or deadline not present in the retrieved policy. (The first call loads the 3B synthesis model; ~30s.)

In [ ]:
queries = [
    'How much is the overdraft fee and can it be reversed?',
    'I want to dispute an unauthorized transaction on my card.',
    'How long do I have to file a dispute?',
]
for q in queries:
    hits = retriever.search(q, k=3)
    if not hits:
        print(f'Q: {q}\n   (no policy routed)\n'); continue
    r = hits[0]
    print(f'Q: {q}')
    print(f'   routed policy : {r["id"]}')
    print(f'   expanded query: {r["expanded_query"]}')
    print(f'   gms context   : {r["gms_context_lines"]} fact lines')
    print(f'   grounded answer: {r["answer"]}')
    print()

**Reading the output.** Each query routes to a single canonical policy via the graph (`overdraft`, `disputes`), and the synthesized answer names that policy and stays inside its text — numbers like the $35 fee or the 60-day dispute window come from the retrieved policy, not from the model's parameters. `gms context` is the number of fact lines the graph walk surfaced before voting on a policy.

## 4. The governed tool

`make_search_policy_tool(policies_dir)` builds the registered `Tool`. Its `fn` lazily constructs the retriever and trims each result body for the agent's context window. This is the object `register_all` adds to the registry.

In [ ]:
from glassloop.capstone.banking_tools import make_search_policy_tool

search_tool = make_search_policy_tool(root / 'data' / 'policies')
print('name   :', search_tool.name)
print('schema :', list(search_tool.input_schema.model_fields),
      '->', list(search_tool.output_schema.model_fields))

out = search_tool.fn(query='Can I get an overdraft fee waiver?')
for r in out['results']:
    print('  id     :', r['id'])
    print('  snippet:', r['text'][:120].replace(chr(10), ' '), '...')
    if r.get('answer'):
        print('  answer :', r['answer'])

## 5. How the store was built

`data/gms_policy_store/` was produced by `scripts/build_policy_rag_store.py` from a single source document, `data/bank_policies.md`:

1. **Ingest** the markdown (policy catalog, per-policy attributes, the alias table, and cross-policy numeric registers) into a GMS store via regex ingestion.
2. **Train** the geometric embeddings (`config.train.epochs = 200`, `lr = 5e-3`) so the graph supports both dense retrieval and multi-hop traversal.
3. **Save** the store (`adapter.json`, `triples.json`, `model.pt`, ENM registers) to `data/gms_policy_store/`.

To rebuild (e.g. after editing `bank_policies.md`):

```bash
python scripts/build_policy_rag_store.py
```

We can peek at the trained store's relations and a few `has_alias` triples — the graph the retriever walks:

In [ ]:
rels = sorted({r for _h, r, _t in retriever.store.triples})
print('relations in the policy store:', rels)
print('\nsample has_alias edges:')
shown = 0
for h, r, t in retriever.store.triples:
    if r == 'has_alias':
        print(f'  {h}  --has_alias-->  {t}')
        shown += 1
        if shown >= 6: break

## Summary

- `search_policy` = **Graph RAG**: a trained kg-memory store routes the query to a canonical policy (alias-expanded, multi-hop), then Qwen synthesizes an answer constrained to the retrieved policy text.
- Grounding is structural: the model only ever sees what the graph returned, so it cannot invent a fee or deadline.
- The store is built and trained by `scripts/build_policy_rag_store.py` from `data/bank_policies.md`; `AGENTLAB_USE_LLM_RAG=0` gives retrieval-only, GPU-free operation.

Next: **Supplement 4 — `flag_regulatory`**, where a trained GMS store verifies and corrects the model's proposed regulatory flags.